# Laporan Performa Pengiriman per Wilayah — Dataset Olist E-Commerce Brasil
**Mata Kuliah Big Data · PRADYTHA GALUH PUTRANTI**

**Pertanyaan bisnis:** Wilayah (negara bagian pelanggan, `customer_state`) mana yang performa pengirimannya paling bermasalah, dan langkah apa yang direkomendasikan untuk memperbaikinya?

**Luaran:** satu tabel ringkasan per wilayah (nama wilayah, jumlah pesanan, rata-rata lama pengiriman, persentase pesanan terlambat) yang disimpan ke CSV.

Sumber data: [Kaggle - Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Alur: **Extract → Transform → Load**.

## 0. Persiapan — memuat data ke Colab
Data ditarik langsung dari Kaggle (tanpa diunduh ke laptop).

In [1]:
!pip install kagglehub -q

import kagglehub, os

DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Folder data:", DATA_DIR)
for f in sorted(os.listdir(DATA_DIR)):
    print(" -", f)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Folder data: /kaggle/input/brazilian-ecommerce
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


## 1. EXTRACT — menarik data dari sumber

Hanya **2 tabel** yang dibutuhkan:
- `orders` → tanggal beli, tanggal diterima pelanggan, tanggal estimasi, status order (1 baris = 1 order)
- `customers` → `customer_state` (wilayah pelanggan)

Tabel `order_items` tidak dipakai karena tabel itu berisi 1 baris per *barang*; kalau dipakai, satu order dengan banyak barang akan terhitung berkali-kali. Karena unit analisis kita adalah **pesanan**, `orders` sudah cukup.

In [2]:
import pandas as pd

orders    = pd.read_csv(os.path.join(DATA_DIR, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(DATA_DIR, 'olist_customers_dataset.csv'))

print(f"orders   : {orders.shape[0]:>7,} baris x {orders.shape[1]} kolom")
print(f"customers: {customers.shape[0]:>7,} baris x {customers.shape[1]} kolom")

orders   :  99,441 baris x 8 kolom
customers:  99,441 baris x 5 kolom


## 2. TRANSFORM — membersihkan, menggabungkan, menurunkan fitur

### 2a. Inspeksi kualitas data
Yang dicek: nilai kosong, duplikat, dan sebaran status order. Kolom tanggal pengiriman punya nilai kosong karena sebagian order belum/tidak sampai ke pelanggan (dibatalkan, masih diproses, dll). Ini ditangani secara eksplisit di langkah penyaringan (2f), bukan dibuang sembarangan.

In [3]:
kolom_tanggal = ['order_purchase_timestamp',
                 'order_delivered_customer_date',
                 'order_estimated_delivery_date']

print("Nilai kosong pada kolom tanggal 'orders':")
print(orders[kolom_tanggal].isnull().sum())

print("\nSebaran status order:")
print(orders['order_status'].value_counts())

print("\nDuplikat baris orders   :", orders.duplicated().sum())
print("Duplikat order_id       :", orders['order_id'].duplicated().sum())
print("Duplikat baris customers:", customers.duplicated().sum())

Nilai kosong pada kolom tanggal 'orders':
order_purchase_timestamp            0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Sebaran status order:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Duplikat baris orders   : 0
Duplikat order_id       : 0
Duplikat baris customers: 0


### 2b. Pembersihan — tipe data tanggal & duplikat
Kolom tanggal masih berupa teks, jadi diubah ke `datetime` agar bisa dihitung selisihnya. `errors='coerce'` membuat nilai yang gagal diubah menjadi `NaT` (kosong) dan tidak menghentikan program.

In [4]:
for c in kolom_tanggal:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

n0 = len(orders)
orders = orders.drop_duplicates()
print(f"Duplikat orders dibuang: {n0 - len(orders)} baris")

n0 = len(customers)
customers = customers.drop_duplicates()
print(f"Duplikat customers dibuang: {n0 - len(customers)} baris")

print("\nTipe data kolom tanggal sekarang:")
print(orders[kolom_tanggal].dtypes)

Duplikat orders dibuang: 0 baris
Duplikat customers dibuang: 0 baris

Tipe data kolom tanggal sekarang:
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


### 2c. Penggabungan (JOIN)
Tempelkan `customer_state` ke tabel `orders` lewat `customer_id`. Dipakai `left join` agar semua order tetap ada, lalu jumlah baris diperiksa supaya tidak bertambah (tanda join yang salah).

In [5]:
orders_wilayah = orders.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id', how='left'
)

# Validasi join: jumlah baris harus sama dengan sebelum join
assert len(orders_wilayah) == len(orders), "Jumlah baris berubah setelah join!"
print("Tabel gabungan:", orders_wilayah.shape[0], "baris x", orders_wilayah.shape[1], "kolom")
print("customer_state kosong setelah join:", orders_wilayah['customer_state'].isnull().sum())
orders_wilayah.head()

Tabel gabungan: 99441 baris x 9 kolom
customer_state kosong setelah join: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP


### 2d. Penurunan fitur — 3 kolom turunan wajib

**Kolom 1 — `lama_kirim_hari`**: selisih hari antara tanggal barang diterima pelanggan (`order_delivered_customer_date`) dan tanggal pembelian (`order_purchase_timestamp`).

In [6]:
# KOLOM TURUNAN 1: lama pengiriman (hari) = tanggal diterima - tanggal beli
orders_wilayah['lama_kirim_hari'] = (
    orders_wilayah['order_delivered_customer_date'] - orders_wilayah['order_purchase_timestamp']
).dt.days

orders_wilayah[['order_purchase_timestamp', 'order_delivered_customer_date', 'lama_kirim_hari']].head()

,order_purchase_timestamp,order_delivered_customer_date,lama_kirim_hari
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


**Kolom 2 — `terlambat`**: `True` jika tanggal diterima **lebih lambat** dari tanggal perkiraan pengiriman, `False` jika tepat waktu/lebih cepat. Untuk order yang belum diterima (tanggal kosong), nilainya dibiarkan kosong (bukan `False`), karena statusnya memang belum bisa ditentukan.

In [7]:
# KOLOM TURUNAN 2: status terlambat (diterima setelah tanggal estimasi)
sudah_diterima = orders_wilayah['order_delivered_customer_date'].notna()

orders_wilayah['terlambat'] = (
    (orders_wilayah['order_delivered_customer_date'] > orders_wilayah['order_estimated_delivery_date'])
    .where(sudah_diterima)      # yang belum diterima -> kosong
    .astype('boolean')
)

print(orders_wilayah['terlambat'].value_counts(dropna=False))

terlambat
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64


**Kolom 3 — `kategori_kecepatan`**: mengelompokkan `lama_kirim_hari` menjadi cepat / sedang / lambat. Ambang ditentukan dari sebaran data, jadi sebaran dilihat dulu:

In [ ]:
# Lihat sebaran lama kirim sebelum menentukan ambang
print(orders_wilayah['lama_kirim_hari'].describe().round(1))
print()
print("Kuartil 1 (25%):", orders_wilayah['lama_kirim_hari'].quantile(0.25))
print("Median   (50%):", orders_wilayah['lama_kirim_hari'].quantile(0.50))
print("Kuartil 3 (75%):", orders_wilayah['lama_kirim_hari'].quantile(0.75))

**Ambang yang dipilih:** cepat ≤ 7 hari (satu minggu), sedang 8–15 hari, lambat > 15 hari (lebih dari dua minggu). Ambang 7 dan 15 dipilih karena berada di sekitar kuartil 1 dan kuartil 3 pada output di atas, sehingga kelas "cepat" dan "lambat" masing-masing mewakili ujung sebaran data, dan kelas "sedang" mewakili bagian tengahnya. Angka bulat 7 dan 15 juga mudah dipahami pihak bisnis.

In [8]:
# KOLOM TURUNAN 3: kategori kecepatan pengiriman
def kategori_kecepatan(hari):
    if pd.isna(hari):
        return 'belum diterima'
    elif hari <= 7:
        return 'cepat'
    elif hari <= 15:
        return 'sedang'
    else:
        return 'lambat'

orders_wilayah['kategori_kecepatan'] = orders_wilayah['lama_kirim_hari'].apply(kategori_kecepatan)

print(orders_wilayah['kategori_kecepatan'].value_counts())
orders_wilayah[['customer_state', 'lama_kirim_hari', 'terlambat', 'kategori_kecepatan']].head(10)

kategori_kecepatan
sedang            39568
cepat             33698
lambat            23210
belum diterima     2965
Name: count, dtype: int64


,customer_state,lama_kirim_hari,terlambat,kategori_kecepatan
0,SP,8.0,False,sedang
1,BA,13.0,False,sedang
2,GO,9.0,False,sedang
3,RN,13.0,False,sedang
4,SP,2.0,False,cepat
5,PR,16.0,False,lambat
6,RS,NaN,<NA>,belum diterima
7,RJ,9.0,False,sedang
8,RS,9.0,False,sedang
9,SP,18.0,False,lambat


### 2e. Penyaringan
Performa pengiriman hanya bisa dihitung untuk order yang **benar-benar sudah sampai ke pelanggan**. Syarat yang dipakai:
1. `order_status == 'delivered'`
2. `lama_kirim_hari` tidak kosong (ada tanggal diterima)
3. `customer_state` tidak kosong

Order yang tidak memenuhi syarat hanya dikeluarkan dari analisis ini, bukan dihapus dari data awal.

In [9]:
total = len(orders_wilayah)

bersih = orders_wilayah[
    (orders_wilayah['order_status'] == 'delivered') &
    (orders_wilayah['lama_kirim_hari'].notna()) &
    (orders_wilayah['customer_state'].notna())
].copy()

print(f"Total order            : {total:,}")
print(f"Siap dianalisis        : {len(bersih):,}")
print(f"Dikeluarkan            : {total - len(bersih):,} ({(total - len(bersih)) / total:.1%})")
print("\nStatus order yang dikeluarkan:")
print(orders_wilayah.loc[~orders_wilayah.index.isin(bersih.index), 'order_status'].value_counts())

# Setelah difilter, semua baris punya nilai terlambat -> aman diubah ke bool biasa
bersih['terlambat'] = bersih['terlambat'].astype(bool)

Total order            : 99,441
Siap dianalisis        : 96,470
Dikeluarkan            : 2,971 (3.0%)

Status order yang dikeluarkan:
order_status
shipped        1107
canceled        625
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


### 2f. Agregasi — tabel ringkasan per wilayah
Satu baris per `customer_state`: jumlah pesanan, rata-rata lama pengiriman, jumlah dan persentase pesanan terlambat.

In [10]:
ringkasan_wilayah = (
    bersih.groupby('customer_state')
    .agg(
        jumlah_pesanan=('order_id', 'nunique'),
        rata2_lama_kirim_hari=('lama_kirim_hari', 'mean'),
        jumlah_terlambat=('terlambat', 'sum'),
    )
    .reset_index()
    .rename(columns={'customer_state': 'wilayah'})
)

ringkasan_wilayah['pct_terlambat'] = (
    ringkasan_wilayah['jumlah_terlambat'] / ringkasan_wilayah['jumlah_pesanan'] * 100
).round(1)
ringkasan_wilayah['rata2_lama_kirim_hari'] = ringkasan_wilayah['rata2_lama_kirim_hari'].round(1)

# Urutkan dari yang paling bermasalah (persentase terlambat tertinggi)
ringkasan_wilayah = ringkasan_wilayah.sort_values('pct_terlambat', ascending=False).reset_index(drop=True)

print("Jumlah wilayah:", len(ringkasan_wilayah))
ringkasan_wilayah

Jumlah wilayah: 27


,wilayah,jumlah_pesanan,rata2_lama_kirim_hari,jumlah_terlambat,pct_terlambat
0,AL,397,24.0,95,23.9
1,MA,717,21.1,141,19.7
2,PI,476,19.0,76,16.0
3,CE,1279,20.8,196,15.3
4,SE,335,21.0,51,15.2
5,BA,3256,18.9,457,14.0
6,RJ,12350,14.8,1664,13.5
7,TO,274,17.2,35,12.8
8,PA,946,23.3,117,12.4
9,RR,41,29.0,5,12.2


Sebaran `kategori_kecepatan` per wilayah (dalam persen) sebagai konteks tambahan:

In [11]:
sebaran_kategori = (
    pd.crosstab(bersih['customer_state'], bersih['kategori_kecepatan'], normalize='index') * 100
).round(1)
sebaran_kategori.loc[ringkasan_wilayah['wilayah']]

kategori_kecepatan,cepat,lambat,sedang
customer_state,,,
AL,2.5,76.1,21.4
MA,2.8,69.3,27.9
PI,4.0,51.7,44.3
CE,4.1,61.8,34.1
SE,3.0,59.4,37.6
BA,5.2,54.0,40.8
RJ,23.9,32.3,43.8
TO,4.0,48.5,47.4
PA,2.5,72.1,25.4


## 3. LOAD — memuat hasil ke tujuan
Tabel ringkasan disimpan sebagai file CSV agar bisa dipakai ulang untuk laporan atau dashboard.

In [12]:
nama_file = 'performa_pengiriman_per_wilayah.csv'
ringkasan_wilayah.to_csv(nama_file, index=False)
print(f"Tersimpan: {nama_file} ({len(ringkasan_wilayah)} baris)")

# Cek balik: baca ulang file untuk memastikan isinya benar
print(pd.read_csv(nama_file).head())

try:
    from google.colab import files
    files.download(nama_file)
except ImportError:
    pass   # bukan di Colab: file sudah tersimpan di folder kerja

Tersimpan: performa_pengiriman_per_wilayah.csv (27 baris)
  wilayah  jumlah_pesanan  rata2_lama_kirim_hari  jumlah_terlambat  \
0      AL             397                   24.0                95   
1      MA             717                   21.1               141   
2      PI             476                   19.0                76   
3      CE            1279                   20.8               196   
4      SE             335                   21.0                51   

   pct_terlambat  
0           23.9  
1           19.7  
2           16.0  
3           15.3  
4           15.2  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Menjawab pertanyaan bisnis
Wilayah dinilai dari dua sisi: **persentase keterlambatan** (ketepatan janji ke pelanggan) dan **rata-rata lama kirim** (kecepatan sebenarnya). Wilayah dengan jumlah pesanan sangat sedikit (< 100) ditandai karena angkanya kurang stabil.

In [13]:
BATAS_SAMPEL = 100
r = ringkasan_wilayah

print("=== 5 wilayah dengan persentase keterlambatan tertinggi ===")
print(r.head(5)[['wilayah','jumlah_pesanan','rata2_lama_kirim_hari','pct_terlambat']].to_string(index=False))

print("\n=== 5 wilayah dengan rata-rata lama kirim terpanjang ===")
print(r.sort_values('rata2_lama_kirim_hari', ascending=False).head(5)
       [['wilayah','jumlah_pesanan','rata2_lama_kirim_hari','pct_terlambat']].to_string(index=False))

kecil = r[r['jumlah_pesanan'] < BATAS_SAMPEL]['wilayah'].tolist()
print(f"\nWilayah dengan < {BATAS_SAMPEL} pesanan (hati-hati menafsirkan): {kecil}")

# Perbandingan dengan rata-rata nasional
nasional_lama = bersih['lama_kirim_hari'].mean()
nasional_late = bersih['terlambat'].mean() * 100
print(f"\nRata-rata seluruh Brasil: {nasional_lama:.1f} hari, {nasional_late:.1f}% terlambat")

=== 5 wilayah dengan persentase keterlambatan tertinggi ===
wilayah  jumlah_pesanan  rata2_lama_kirim_hari  pct_terlambat
     AL             397                   24.0           23.9
     MA             717                   21.1           19.7
     PI             476                   19.0           16.0
     CE            1279                   20.8           15.3
     SE             335                   21.0           15.2

=== 5 wilayah dengan rata-rata lama kirim terpanjang ===
wilayah  jumlah_pesanan  rata2_lama_kirim_hari  pct_terlambat
     RR              41                   29.0           12.2
     AP              67                   26.7            4.5
     AM             145                   26.0            4.1
     AL             397                   24.0           23.9
     PA             946                   23.3           12.4

Wilayah dengan < 100 pesanan (hati-hati menafsirkan): ['RR', 'AP', 'AC']

Rata-rata seluruh Brasil: 12.1 hari, 8.1% terlambat


In [14]:
# Kesimpulan otomatis dari hasil run
top_late = r.iloc[0]
top_lama = r.sort_values('rata2_lama_kirim_hari', ascending=False).iloc[0]
besar = r[r['jumlah_pesanan'] >= BATAS_SAMPEL]
top_besar = besar.iloc[0]

print("KESIMPULAN")
print(f"1. Persentase keterlambatan tertinggi: {top_late['wilayah']} "
      f"({top_late['pct_terlambat']}% dari {int(top_late['jumlah_pesanan'])} pesanan; "
      f"rata-rata nasional {nasional_late:.1f}%).")
print(f"2. Rata-rata lama kirim terpanjang: {top_lama['wilayah']} "
      f"({top_lama['rata2_lama_kirim_hari']} hari; rata-rata nasional {nasional_lama:.1f} hari).")
print(f"3. Di antara wilayah dengan >= {BATAS_SAMPEL} pesanan (angka lebih stabil), "
      f"yang paling terlambat: {top_besar['wilayah']} "
      f"({top_besar['pct_terlambat']}% terlambat, {top_besar['rata2_lama_kirim_hari']} hari).")

KESIMPULAN
1. Persentase keterlambatan tertinggi: AL (23.9% dari 397 pesanan; rata-rata nasional 8.1%).
2. Rata-rata lama kirim terpanjang: RR (29.0 hari; rata-rata nasional 12.1 hari).
3. Di antara wilayah dengan >= 100 pesanan (angka lebih stabil), yang paling terlambat: AL (23.9% terlambat, 24.0 hari).


### Rekomendasi
Rekomendasi ditujukan untuk wilayah teratas pada tabel `ringkasan_wilayah` (persentase terlambat tertinggi) dan wilayah dengan lama kirim terpanjang pada hasil di atas.

1. **Evaluasi ulang tanggal estimasi pengiriman** untuk wilayah tersebut. Jika estimasi yang dijanjikan terlalu optimis dibanding kondisi logistik riil, persentase "terlambat" tinggi meskipun waktu tempuhnya wajar untuk jaraknya. Menyesuaikan estimasi adalah perbaikan tercepat dan termurah.
2. **Dekatkan stok ke pelanggan** lewat gudang / *fulfillment center* regional atau kerja sama dengan penjual di sekitar wilayah tersebut, untuk memangkas jarak tempuh dan waktu *last-mile*.
3. **Evaluasi mitra kurir/ekspedisi** yang melayani rute wilayah tersebut, misalnya membandingkan performa antar kurir jika datanya tersedia, lalu menambah atau mengganti mitra pada rute yang konsisten lambat.
4. **Pantau rutin** dengan tabel ini (persentase terlambat dan rata-rata lama kirim per wilayah) setiap bulan untuk melihat apakah perbaikan berhasil.

**Catatan keterbatasan:** analisis ini hanya memakai order berstatus `delivered`, sehingga order yang dibatalkan atau tidak pernah sampai tidak tercermin dalam angka keterlambatan. Wilayah dengan jumlah pesanan sedikit bisa punya persentase yang bergejolak.